# 테스트 전 토큰화 함수 및 임폴트

In [2]:
from sklearn.metrics import classification_report
import pandas as pd  # pandas 누락
from kiwipiepy import Kiwi  # kiwi 토크나이저
import joblib  #  모델 불러오기


# ✅ tokenizer 함수 정의

kiwi = Kiwi()

def tokenize_and_filter(text):
    result = kiwi.analyze(text)[0][0]
    tokens = []
    for word, pos, _, _ in result:
        if pos in {"NNG", "NNP", "VV", "VA"}:
            if pos in {"VV", "VA"}:
                word = word + "다"
            tokens.append(word)
    return " ".join(tokens)

In [ ]:
# # --- 0. 필요한 라이브러리 모두 불러오기 ---
# import pandas as pd
# import numpy as np
# import re
# import os
# import joblib
# from typing import List

# from kiwipiepy import Kiwi
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.model_selection import StratifiedKFold, cross_val_predict
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import classification_report

# # 스태킹에 사용할 모델들
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import (
#     StackingClassifier,
#     RandomForestClassifier,
# )

# # --- ✅ 1. 설정 (새로운 데이터셋에 맞게 이 부분만 수정하세요) ---

# # 1-1. 파일 및 열 이름 설정
# DATASET_PATH = "your_new_dataset.csv"  # 👈 여기에 새 데이터셋 파일 경로를 입력
# TEXT_COLUMN = "text"  # 👈 텍스트가 들어있는 열 이름
# LABEL_COLUMN = "is_phishing"  # 👈 정답 라벨이 들어있는 열 이름

# # 1-2. 저장될 최종 모델의 경로와 이름 설정
# MODEL_SAVE_DIR = "models"
# MODEL_SAVE_NAME = "new_stacking_model.pkl"
# MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, MODEL_SAVE_NAME)


# # --- 2. 토크나이저 클래스 정의 (최종본) ---
# class KiwiTokenizer:
#     # (이전과 동일한 최종 클래스 정의... 생략하지 않고 모두 포함)
#     def __init__(self):
#         self.kiwi = None
#         self.stop_words = ["하", "있", "되"]

#     def __call__(self, text: str) -> List[str]:
#         if self.kiwi is None:
#             self.kiwi = Kiwi()
#             self.kiwi.add_user_word("대포통장", "NNP")
#             self.kiwi.add_user_word("계좌번호", "NNP")

#         cleaned_text = re.sub(r"[^\w\s<>]", " ", text)
#         cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

#         tokens = self.kiwi.tokenize(cleaned_text)
#         result_tokens = [
#             token.form
#             for token in tokens
#             if token.tag in ["NNG", "NNP", "VV", "VA", "XR"]
#             and token.form not in self.stop_words
#         ]
#         return result_tokens

#     def __getstate__(self):
#         state = self.__dict__.copy()
#         if "kiwi" in state:
#             del state["kiwi"]
#         return state

#     def __setstate__(self, state):
#         self.__dict__.update(state)
#         self.kiwi = None


# # --- 3. 데이터 불러오기 ---
# print(f"'{DATASET_PATH}' 데이터를 불러옵니다...")
# try:
#     df = pd.read_csv(DATASET_PATH)
#     # 설정된 열 이름을 사용해 X, y 데이터 지정
#     X = df[TEXT_COLUMN].astype(str)
#     y = df[LABEL_COLUMN]
#     print(f"✅ 총 {len(df)}개의 데이터 로드 완료.")
# except FileNotFoundError:
#     print(f"🚨 파일을 찾을 수 없습니다: '{DATASET_PATH}'")
#     exit()
# except KeyError as e:
#     print(f"🚨 CSV 파일에 필요한 열({e})이 없습니다. 설정 부분을 확인해주세요.")
#     exit()

# # --- 4. 스태킹 모델 및 파이프라인 구성 ---
# print("\n스태킹 모델과 파이프라인을 구성합니다...")
# base_models = [
#     ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
#     (
#         "rf",
#         RandomForestClassifier(
#             n_estimators=100, class_weight="balanced", random_state=42
#         ),
#     ),
# ]
# meta_model = LogisticRegression(max_iter=1000, random_state=42)
# stacked_clf = StackingClassifier(
#     estimators=base_models, final_estimator=meta_model, cv=5, n_jobs=-1
# )

# pipeline_stacking = Pipeline(
#     [
#         (
#             "tfidf",
#             TfidfVectorizer(
#                 tokenizer=KiwiTokenizer(),
#                 sublinear_tf=True,
#                 min_df=5,
#                 max_df=0.9,
#                 ngram_range=(1, 2),
#             ),
#         ),
#         ("clf", stacked_clf),
#     ]
# )
# print("✅ 파이프라인 구성 완료.")

# # --- 5. K-Fold 교차 검증으로 성능 평가 ---
# print("\nK-Fold 교차 검증을 시작합니다 (데이터 양에 따라 시간이 소요될 수 있습니다)...")
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# y_pred_stack = cross_val_predict(pipeline_stacking, X, y, cv=skf, n_jobs=-1)
# print("✅ 교차 검증 완료.")
# print("\n--- [새로운 데이터셋 기반 모델 성능 리포트] ---")
# print(classification_report(y, y_pred_stack, target_names=["정상대화", "보이스피싱"]))
# print("---------------------------------------------")

# # --- 6. 전체 데이터로 최종 모델 학습 및 저장 ---
# print("\n전체 데이터로 최종 모델을 학습하고 저장합니다...")
# pipeline_stacking.fit(X, y)
# os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
# joblib.dump(pipeline_stacking, MODEL_SAVE_PATH)
# print(f"✅ 최종 모델이 '{MODEL_SAVE_PATH}' 파일로 저장되었습니다.")

In [ ]:
# # --- 1. 파일 경로 및 모델 로딩 ---
# print("저장된 스태킹 모델 파이프라인을 불러옵니다...")

# pipeline_path = os.path.join("models", "stacking_model_pipeline_6000_JJ.pkl")

# try:
#     loaded_pipeline = joblib.load(pipeline_path)
#     print(f"모델 로드 완료: {pipeline_path}")
# except Exception as e:
#     print(f"모델 로딩 중 에러 발생: {e}")

In [ ]:
# # --- 2. 예측, 평가, 결과 출력 함수 정의 ---
# def predict_and_display_results(csv_path: str, pipeline):
#     """
#     주어진 CSV 파일을 읽어 피싱 여부를 예측하고,
#     정답지가 있을 경우 성능 평가까지 수행하는 함수.
#     """
#     print("\n" + "#" * 60)
#     print(f"'{csv_path}' 파일에 대한 분석을 시작합니다.")

#     try:
#         df = pd.read_csv(csv_path)
#     except FileNotFoundError:
#         print(f"파일을 찾을 수 없습니다: '{csv_path}'")
#         return

#     if "text" not in df.columns:
#         print(f"'{csv_path}' 파일에 'text' 열이 없습니다.")
#         return

#     X_new = df["text"].astype(str)

#     # 예측 수행
#     predictions = pipeline.predict(X_new)
#     probabilities = pipeline.predict_proba(X_new)

#     # 예측 결과를 데이터프레임에 추가
#     df["predicted_label"] = predictions
#     df["phishing_probability"] = probabilities[:, 1] * 100

#     # 예측 결과 요약 출력
#     print("\n[ 예측 결과 요약 ]")
#     prediction_counts = df["predicted_label"].value_counts()
#     print(f" - 정상 대화 (0) 예측: {prediction_counts.get(0, 0)}건")
#     print(f" - 보이스피싱 (1) 예측: {prediction_counts.get(1, 0)}건")

#     # 성능 평가 로직 추가
#     # 'is_phishing' 정답 열이 있을 경우에만 평가 수행
#     if "is_phishing" in df.columns:
#         y_true = df["is_phishing"]

#         print("\n--- [ 모델 성능 평가 리포트 ] ---")
#         # Classification Report 출력 (정밀도, 재현율, f1-score)
#         print(
#             classification_report(
#                 y_true, predictions, target_names=["정상대화", "보이스피싱"]
#             )
#         )

#         # 혼동 행렬 (Confusion Matrix) 시각화
#         cm = confusion_matrix(y_true, predictions)
#         plt.figure(figsize=(8, 6))
#         sns.heatmap(
#             cm,
#             annot=True,
#             fmt="d",
#             cmap="Blues",
#             xticklabels=["정상대화 (예측)", "보이스피싱 (예측)"],
#             yticklabels=["정상대화 (실제)", "보이스피싱 (실제)"],
#         )
#         plt.title(f"[{csv_path}] 혼동 행렬", fontsize=16)
#         plt.ylabel("실제 값", fontsize=12)
#         plt.xlabel("예측 값", fontsize=12)
#         plt.show()  # 그래프 보여주기
#     else:
#         print("\n[알림] 정답(is_phishing) 열이 없어 성능 평가는 생략합니다.")

#     print("\n[ 예측 결과 샘플 (상위 10개) ]")
#     display_columns = ["text", "predicted_label", "phishing_probability"]
#     if "is_phishing" in df.columns:
#         display_columns.insert(1, "is_phishing")

#     pd.options.display.float_format = "{:,.2f}".format
#     display(df[display_columns].head(10))
#     print("#" * 60)


# # --- 3. 테스트할 파일 목록에 대해 분석 실행 ---
# files_to_test = ["1차모델_테스트데이터셋.csv", "시나리오통화테스트셋.csv"]

# for file_path in files_to_test:
#     predict_and_display_results(file_path, loaded_pipeline)

# 앙상블 Stacking 테스트

In [3]:
from sklearn.metrics import classification_report

# 테스트셋 불러오기
test_df = pd.read_csv("../../dataset/시나리오통화테스트셋.csv")  # 실제 경로로 바꿔줘
test_df["processed_text"] = test_df["text"].astype(str).apply(tokenize_and_filter)

X_test = test_df["processed_text"]
y_test = test_df["is_phishing"]

# 저장된 모델 불러오기
import joblib

model = joblib.load("../../models/pipeline_stacking.pkl")

# 예측 및 성능 확인
y_pred_test = model.predict(X_test)
print(classification_report(y_test, y_pred_test))

# 예측 및 확률 추론
y_pred_proba = model.predict_proba(X_test)[:, 1]  # 클래스 1(피싱)일 확률

# 확률 포함한 결과 데이터프레임 출력 (선택적으로 저장 가능)
results_df = test_df.copy()
results_df["predicted_label"] = y_pred_test
results_df["phishing_probability"] = y_pred_proba

# 확률 상위 5개 예시 출력
print("\n🔍 예측 확률 예시:")
print(
    results_df[
        ["text", "is_phishing", "predicted_label", "phishing_probability"]
    ].head(20)
)

              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.91      1.00      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20


🔍 예측 확률 예시:
                                                 text  is_phishing  \
0   네, 여보세요.\n네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니...            1   
1   고객님, 안녕하세요. 농협캐피탈입니다.\n고객님은 신용등급 상향 대상자로 선정되셔서...            1   
2   여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어.\n누구...            1   
3   네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 ...            1   
4   네, 안녕하세요. 국민건강보험공단입니다. 고객님.\n작년 과오납 환급금이 47만원 ...            1   
5   네, 여보세요.\n네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요?\...            0   
6   엄마, 나 또 액정이 깨졌어, 지금 친구 폰으로 전화했어.\n더 깨졌어, 핸드폰을 ...            0   
7   어제 주문한 물건이 계속 배송 중으로만 떠있어서요.\n운송장 번호 확인 가능하실까요...            0   
8   세종내과입니다. 무엇을 도와드릴까요?\n혹시 그

# 단순 로지스틱 회귀

In [9]:
from sklearn.metrics import classification_report

# 테스트셋 불러오기
test_df = pd.read_csv("../../dataset/시나리오통화테스트셋.csv")  # 실제 경로로 바꿔줘
test_df["processed_text"] = test_df["text"].astype(str).apply(tokenize_and_filter)

X_test = test_df["processed_text"]
y_test = test_df["is_phishing"]

# 저장된 모델 불러오기
import joblib

model = joblib.load("../../models/1차모델_원본데이터_pipeline_로지스틱회귀.pkl")

# 예측 및 성능 확인
y_pred_test = model.predict(X_test)
print(classification_report(y_test, y_pred_test))

# 예측 및 확률 추론
y_pred_proba = model.predict_proba(X_test)[:, 1]  # 클래스 1(피싱)일 확률

# 확률 포함한 결과 데이터프레임 출력 (선택적으로 저장 가능)
results_df = test_df.copy()
results_df["predicted_label"] = y_pred_test
results_df["phishing_probability"] = y_pred_proba

# 확률 상위 5개 예시 출력
print("\n🔍 예측 확률 예시 :")
print(
    results_df[
        ["text", "is_phishing", "predicted_label", "phishing_probability"]
    ].head(20)
)

              precision    recall  f1-score   support

           0       1.00      0.70      0.82        10
           1       0.77      1.00      0.87        10

    accuracy                           0.85        20
   macro avg       0.88      0.85      0.85        20
weighted avg       0.88      0.85      0.85        20


🔍 예측 확률 예시 :
                                                 text  is_phishing  \
0   네, 여보세요.\n네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니...            1   
1   고객님, 안녕하세요. 농협캐피탈입니다.\n고객님은 신용등급 상향 대상자로 선정되셔서...            1   
2   여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어.\n누구...            1   
3   네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 ...            1   
4   네, 안녕하세요. 국민건강보험공단입니다. 고객님.\n작년 과오납 환급금이 47만원 ...            1   
5   네, 여보세요.\n네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요?\...            0   
6   엄마, 나 또 액정이 깨졌어, 지금 친구 폰으로 전화했어.\n더 깨졌어, 핸드폰을 ...            0   
7   어제 주문한 물건이 계속 배송 중으로만 떠있어서요.\n운송장 번호 확인 가능하실까요...            0   
8   세종내과입니다. 무엇을 도와드릴까요?\n혹시 

# Ngram, kfold 적용 로지스틱회귀

In [10]:
from sklearn.metrics import classification_report

# 테스트셋 불러오기
test_df = pd.read_csv("../../dataset/시나리오통화테스트셋.csv")  # 실제 경로로 바꿔줘
test_df["processed_text"] = test_df["text"].astype(str).apply(tokenize_and_filter)

X_test = test_df["processed_text"]
y_test = test_df["is_phishing"]

# 저장된 모델 불러오기
import joblib

model = joblib.load("../../models/1차모델_원본데이터_pipeline_Ngram_kfold_로지스틱회귀.pkl")

# 예측 및 성능 확인
y_pred_test = model.predict(X_test)
print(classification_report(y_test, y_pred_test))

# 예측 및 확률 추론
y_pred_proba = model.predict_proba(X_test)[:, 1]  # 클래스 1(피싱)일 확률

# 확률 포함한 결과 데이터프레임 출력 (선택적으로 저장 가능)
results_df = test_df.copy()
results_df["predicted_label"] = y_pred_test
results_df["phishing_probability"] = y_pred_proba

# 확률 상위 5개 예시 출력
print("\n🔍 예측 확률 예시 :")
print(
    results_df[
        ["text", "is_phishing", "predicted_label", "phishing_probability"]
    ].head(20)
)

              precision    recall  f1-score   support

           0       1.00      0.70      0.82        10
           1       0.77      1.00      0.87        10

    accuracy                           0.85        20
   macro avg       0.88      0.85      0.85        20
weighted avg       0.88      0.85      0.85        20


🔍 예측 확률 예시 :
                                                 text  is_phishing  \
0   네, 여보세요.\n네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니...            1   
1   고객님, 안녕하세요. 농협캐피탈입니다.\n고객님은 신용등급 상향 대상자로 선정되셔서...            1   
2   여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어.\n누구...            1   
3   네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 ...            1   
4   네, 안녕하세요. 국민건강보험공단입니다. 고객님.\n작년 과오납 환급금이 47만원 ...            1   
5   네, 여보세요.\n네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요?\...            0   
6   엄마, 나 또 액정이 깨졌어, 지금 친구 폰으로 전화했어.\n더 깨졌어, 핸드폰을 ...            0   
7   어제 주문한 물건이 계속 배송 중으로만 떠있어서요.\n운송장 번호 확인 가능하실까요...            0   
8   세종내과입니다. 무엇을 도와드릴까요?\n혹시 

# Bagging(랜덤 포레스트) 모델

In [11]:
from sklearn.metrics import classification_report

# 테스트셋 불러오기
test_df = pd.read_csv("../../dataset/시나리오통화테스트셋.csv")  # 실제 경로로 바꿔줘
test_df["processed_text"] = test_df["text"].astype(str).apply(tokenize_and_filter)

X_test = test_df["processed_text"]
y_test = test_df["is_phishing"]

# 저장된 모델 불러오기
import joblib

model = joblib.load("../../models/1차모델_원본데이터_pipeline_rf_Bagging.pkl")

# 예측 및 성능 확인
y_pred_test = model.predict(X_test)
print(classification_report(y_test, y_pred_test))

# 예측 및 확률 추론
y_pred_proba = model.predict_proba(X_test)[:, 1]  # 클래스 1(피싱)일 확률

# 확률 포함한 결과 데이터프레임 출력 (선택적으로 저장 가능)
results_df = test_df.copy()
results_df["predicted_label"] = y_pred_test
results_df["phishing_probability"] = y_pred_proba

# 확률 상위 5개 예시 출력
print("\n🔍 예측 확률 예시 :")
print(
    results_df[
        ["text", "is_phishing", "predicted_label", "phishing_probability"]
    ].head(20)
)

              precision    recall  f1-score   support

           0       1.00      0.80      0.89        10
           1       0.83      1.00      0.91        10

    accuracy                           0.90        20
   macro avg       0.92      0.90      0.90        20
weighted avg       0.92      0.90      0.90        20


🔍 예측 확률 예시 :
                                                 text  is_phishing  \
0   네, 여보세요.\n네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니...            1   
1   고객님, 안녕하세요. 농협캐피탈입니다.\n고객님은 신용등급 상향 대상자로 선정되셔서...            1   
2   여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어.\n누구...            1   
3   네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 ...            1   
4   네, 안녕하세요. 국민건강보험공단입니다. 고객님.\n작년 과오납 환급금이 47만원 ...            1   
5   네, 여보세요.\n네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요?\...            0   
6   엄마, 나 또 액정이 깨졌어, 지금 친구 폰으로 전화했어.\n더 깨졌어, 핸드폰을 ...            0   
7   어제 주문한 물건이 계속 배송 중으로만 떠있어서요.\n운송장 번호 확인 가능하실까요...            0   
8   세종내과입니다. 무엇을 도와드릴까요?\n혹시 